# DBTS Variants Bake-off (no regime classifier)

Regressor fixed: **NS_spline rolling (win=200, decay=0.995)** — winner of Regression_Bakeoff.

Universe: 10 sectors, each with anchor + 10 peers = 11 candidates.

**3 orthogonal axes → 18 combinations**:
1. Score: `|rz|` | `|rz|*quality` | `quality_only`
2. Universe: `all_11` | `anchors_top3` (3 most stationary per sector on TRAIN)
3. Stickiness: `none` | `soft (min_hold=5, margin=0.10)` | `hard (min_hold=20, ratio=0.90)`

Quality = `0.5*mr_hit_rate + 0.3*pred_return_ic + 0.2*(1-adf_p)` (all from TRAIN only).
PM: simple residual-z reversion. ENTER if `|rz|>=1.5` (LONG if rz<0). EXIT on `|rz|<=0.3`, `max_hold=10`, `stop=-2%`, `take=+3%`.
Costs 5 bps. One position per sector.

In [ ]:
# Cell 2 — Imports + setup
import os, sys, math, pickle, warnings, time
import random as _random
from pathlib import Path
from datetime import datetime
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.preprocessing import StandardScaler, SplineTransformer
try:
    from statsmodels.tsa.stattools import adfuller
except Exception:
    adfuller = None
warnings.filterwarnings('ignore')
%matplotlib inline
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'strategy').exists():
    cand = Path(r'C:\algo-trading-project')
    if cand.exists(): PROJECT_ROOT = cand
sys.path.insert(0, str(PROJECT_ROOT)); os.chdir(PROJECT_ROOT)
SEED = 42
_random.seed(SEED); np.random.seed(SEED)
FORCE_RECOMPUTE = False
CACHE_DIR = Path('outputs/dbts_variants'); CACHE_DIR.mkdir(parents=True, exist_ok=True)
REG_CACHE     = CACHE_DIR / 'ns_regressors.pkl'
QUALITY_CACHE = CACHE_DIR / 'quality.pkl'
print(f'cache: {CACHE_DIR}')

In [ ]:
# Cell 3 — Data, splits, sector pools
from config import SECTORS
from strategy.strategy_config import StrategyConfig
from strategy.pipeline import StrategyPipeline
from strategy.splits import chrono_split
cfg = StrategyConfig(force_recompute=False, make_plots=False)
pipeline = StrategyPipeline(cfg)
md = pipeline.load_data()
sp = chrono_split(md.prices.index, cfg)
train_idx = pd.DatetimeIndex(sp.train_idx).sort_values()
val_idx   = pd.DatetimeIndex(sp.val_idx).sort_values()
test_idx  = pd.DatetimeIndex(sp.test_idx).sort_values()
SECTOR_POOLS = {}
for etf, s in SECTORS.items():
    pool = [s['target']] + list(s['predictors'])
    pool = [t for t in pool if t in md.prices.columns]
    SECTOR_POOLS[s['name']] = pool
print(f'TRAIN n={len(train_idx)} | VAL n={len(val_idx)} | TEST n={len(test_idx)}')
print(f'sectors: {len(SECTOR_POOLS)}, total candidates: {sum(len(p) for p in SECTOR_POOLS.values())}')

In [ ]:
# Cell 4 — NS_spline rolling WF regressor + cached run per (sector, candidate)
RETRAIN_EVERY=50; MIN_TRAIN_DAYS=100; ROLL_WIN=200
DECAY_ALPHA=0.995; TOP_K_FEATS=5
SPLINE_K=4; SPLINE_DEG=3; RIDGE_ALPHA=1.0
EN_ALPHA=0.01; EN_L1_RATIO=0.5; RESID_Z_WIN=60

def decay_weights(n, alpha=DECAY_ALPHA):
    w = alpha ** np.arange(n-1,-1,-1, dtype=float); return w / w.sum() * n

def feature_select(X, y, w, top_k=TOP_K_FEATS):
    sc = StandardScaler().fit(X)
    en = ElasticNet(alpha=EN_ALPHA, l1_ratio=EN_L1_RATIO, max_iter=5000, random_state=SEED)
    en.fit(sc.transform(X), y, sample_weight=w)
    coef = np.abs(en.coef_)
    if not np.any(coef > 0): return np.arange(min(top_k, X.shape[1]))
    return np.argsort(-coef)[:top_k]

def fit_ns(X, y, w):
    sc = StandardScaler().fit(X); Xs = sc.transform(X)
    sp_ = SplineTransformer(n_knots=SPLINE_K, degree=SPLINE_DEG, knots='quantile',
                            extrapolation='linear').fit(Xs)
    Z = sp_.transform(Xs)
    m = Ridge(alpha=RIDGE_ALPHA, random_state=SEED).fit(Z, y, sample_weight=w)
    return lambda Xq: m.predict(sp_.transform(sc.transform(Xq)))

def wf_ns(X_df, y_ser):
    idx = X_df.index; n = len(idx)
    pred1 = np.full(n, np.nan)
    last_refit = -10**9; predict_fn = None; sel_idx = None
    Xv = X_df.values; yv = y_ser.values
    for t in range(n):
        if t >= MIN_TRAIN_DAYS and (t - last_refit) >= RETRAIN_EVERY:
            lo = max(0, t - ROLL_WIN)
            Xt = Xv[lo:t]; yt = yv[lo:t]
            mask = (~np.isnan(Xt).any(axis=1)) & (~np.isnan(yt))
            if mask.sum() < MIN_TRAIN_DAYS: continue
            Xt = Xt[mask]; yt = yt[mask]
            w = decay_weights(len(yt))
            try:
                sel_idx = feature_select(Xt, yt, w)
                predict_fn = fit_ns(Xt[:, sel_idx], yt, w)
                last_refit = t
            except Exception:
                predict_fn = None
        if predict_fn is not None and sel_idx is not None and not np.isnan(Xv[t]).any():
            try: pred1[t] = float(predict_fn(Xv[t:t+1, sel_idx])[0])
            except Exception: pass
    return pd.Series(pred1, index=idx)

if REG_CACHE.exists() and not FORCE_RECOMPUTE:
    with open(REG_CACHE, 'rb') as f: REG = pickle.load(f)
    print(f'[CACHE] regressors: {len(REG)} pairs')
else:
    print('[RECOMPUTE] NS_spline rolling for all (sector, candidate)...')
    REG = {}
    t0 = time.time()
    for sector, pool in SECTOR_POOLS.items():
        for cand in pool:
            peers = [p for p in pool if p != cand]
            if not peers: continue
            Xp = md.prices[peers].copy(); yp = md.prices[cand].copy()
            mask = (~Xp.isna().any(axis=1)) & yp.notna()
            X = Xp[mask]; y = yp[mask]
            if len(y) < MIN_TRAIN_DAYS + 50: continue
            pred = wf_ns(X, y).reindex(md.prices.index)
            price = md.prices[cand].reindex(md.prices.index)
            residual = price - pred
            resz = (residual - residual.rolling(RESID_Z_WIN).mean()) / residual.rolling(RESID_Z_WIN).std()
            pr   = (pred.shift(-1) / price - 1.0)
            nxt  = price.pct_change().shift(-1)
            REG[(sector, cand)] = pd.DataFrame({
                'price': price, 'pred': pred, 'residual': residual,
                'residual_z': resz, 'predicted_return': pr, 'next_ret': nxt,
            })
        print(f'  {sector:18s} done ({sum(1 for k in REG if k[0]==sector)} cands)')
    with open(REG_CACHE, 'wb') as f: pickle.dump(REG, f)
    print(f'Saved -> {REG_CACHE.name} ({time.time()-t0:.1f}s, {len(REG)} pairs)')

In [ ]:
# Cell 5 — Compute QUALITY per (sector, candidate) on TRAIN only
# quality = 0.5*mr_hit + 0.3*pred_ret_ic + 0.2*(1-adf_p), each normalized to [0,1]
RZ_ENTRY_FOR_HIT = 1.5
RZ_EXIT_FOR_HIT  = 0.5
HIT_WINDOW       = 10
IC_HORIZON       = 5

def safe_adf(s, fallback=1.0):
    s = pd.Series(s).dropna()
    if len(s) < 30 or adfuller is None: return fallback
    try:    return float(adfuller(s, autolag='AIC')[1])
    except Exception: return fallback

def mr_hit_rate(rz, win=HIT_WINDOW, enter=RZ_ENTRY_FOR_HIT, exitt=RZ_EXIT_FOR_HIT):
    rz = rz.dropna().values
    if len(rz) < win + 10: return np.nan
    cnt = 0; hits = 0
    i = 0
    while i < len(rz) - win:
        if abs(rz[i]) >= enter:
            cnt += 1
            window = rz[i+1:i+1+win]
            if np.any(np.abs(window) <= exitt):
                hits += 1
            i += win  # don't double count
        else:
            i += 1
    return hits / cnt if cnt > 0 else np.nan

def pred_return_ic(pr, next_ret_h):
    df = pd.DataFrame({'p': pr, 'n': next_ret_h}).dropna()
    if len(df) < 30: return np.nan
    rho, _ = spearmanr(df['p'].values, df['n'].values)
    return float(rho) if np.isfinite(rho) else np.nan

if QUALITY_CACHE.exists() and not FORCE_RECOMPUTE:
    with open(QUALITY_CACHE, 'rb') as f: QUAL = pickle.load(f)
    print(f'[CACHE] quality: {len(QUAL)} pairs')
else:
    print('[RECOMPUTE] quality scores on TRAIN...')
    raws = {}
    for key, df in REG.items():
        sub_train = df.loc[df.index.isin(train_idx)]
        rz_tr = sub_train['residual_z']
        pr_tr = sub_train['predicted_return']
        price = sub_train['price']
        next_ret_h = price.pct_change(IC_HORIZON).shift(-IC_HORIZON)
        raws[key] = {
            'mr_hit': mr_hit_rate(rz_tr),
            'pred_ic': pred_return_ic(pr_tr, next_ret_h),
            'adf_p':  safe_adf(sub_train['residual']),
        }
    raw_df = pd.DataFrame(raws).T
    def norm01(x):
        x = x.copy().astype(float)
        if x.notna().sum() == 0: return x.fillna(0.5)
        lo, hi = float(np.nanmin(x)), float(np.nanmax(x))
        if hi == lo: return pd.Series(0.5, index=x.index)
        return (x - lo) / (hi - lo)
    n_hit  = norm01(raw_df['mr_hit'])
    n_ic   = norm01(raw_df['pred_ic'])
    n_stab = norm01(1.0 - raw_df['adf_p'].clip(0,1))
    quality = (0.5 * n_hit + 0.3 * n_ic + 0.2 * n_stab).fillna(0.0)
    QUAL = {k: {'quality': float(quality[k]),
                'mr_hit': float(raw_df.loc[k,'mr_hit']) if pd.notna(raw_df.loc[k,'mr_hit']) else np.nan,
                'pred_ic': float(raw_df.loc[k,'pred_ic']) if pd.notna(raw_df.loc[k,'pred_ic']) else np.nan,
                'adf_p': float(raw_df.loc[k,'adf_p'])}
            for k in raw_df.index}
    with open(QUALITY_CACHE, 'wb') as f: pickle.dump(QUAL, f)
    print(f'Saved -> {QUALITY_CACHE.name}')

qual_df = pd.DataFrame(QUAL).T
qual_df.index = pd.MultiIndex.from_tuples(qual_df.index, names=['sector','candidate'])
print('\nTop-5 candidates by quality per sector:')
for sector in SECTOR_POOLS:
    sub = qual_df.loc[sector].sort_values('quality', ascending=False).head(5)
    print(f'\n{sector}:')
    print(sub.round(3).to_string())

In [ ]:
# Cell 6 — Build per-sector candidate stacks (resz, pr, next, price) for fast PM
STACKS = {}
for sector, pool in SECTOR_POOLS.items():
    cands = [c for c in pool if (sector, c) in REG]
    if not cands: continue
    idx = md.prices.index
    resz = pd.DataFrame({c: REG[(sector,c)]['residual_z']       for c in cands}).reindex(idx)
    pr   = pd.DataFrame({c: REG[(sector,c)]['predicted_return'] for c in cands}).reindex(idx)
    nxt  = pd.DataFrame({c: REG[(sector,c)]['next_ret']         for c in cands}).reindex(idx)
    prc  = pd.DataFrame({c: REG[(sector,c)]['price']            for c in cands}).reindex(idx)
    quality_vec = np.array([QUAL[(sector,c)]['quality'] for c in cands])
    # anchors_top3: indices of top-3 quality candidates
    top3_ix = np.argsort(-quality_vec)[:3]
    STACKS[sector] = dict(cands=cands, resz=resz, pr=pr, nxt=nxt, prc=prc,
                          quality=quality_vec, top3_ix=top3_ix)
    top3_names = [cands[i] for i in top3_ix]
    print(f'{sector:18s} top3_anchors: {top3_names}')

In [ ]:
# Cell 7 — DBTS selector: 3 score functions × 2 universes × 3 stickiness levels
def build_panel(sector, score_fn, universe, stick):
    """Return decision panel (one row per date) for this sector.
    score_fn  : 'rz' | 'rz_x_q' | 'q'
    universe  : 'all' | 'top3'
    stick     : 'none' | 'soft' | 'hard'  (soft=5d/0.10 abs, hard=20d/ratio 0.90)
    """
    st = STACKS[sector]
    cands = st['cands']; idx = st['resz'].index
    # restrict universe
    if universe == 'top3':
        keep = st['top3_ix']
    else:
        keep = np.arange(len(cands))
    cand_arr = np.array(cands)[keep]
    resz = st['resz'].values[:, keep]
    pr   = st['pr'].values[:, keep]
    nxt  = st['nxt'].values[:, keep]
    prc  = st['prc'].values[:, keep]
    qv   = st['quality'][keep]
    abs_rz = np.abs(np.nan_to_num(resz, nan=0.0))
    if score_fn == 'rz':       scores = abs_rz
    elif score_fn == 'rz_x_q': scores = abs_rz * qv[None, :]
    elif score_fn == 'q':      scores = np.broadcast_to(qv[None, :], abs_rz.shape).copy()
    else: raise ValueError(score_fn)
    valid = (~np.isnan(resz))
    scores = np.where(valid, scores, -np.inf)
    n = len(idx)
    if stick == 'none':
        winners = np.argmax(scores, axis=1)
    else:
        min_hold = 5 if stick == 'soft' else 20
        win = np.empty(n, dtype=int); win[:] = -1
        cur = -1; held = 0
        for i in range(n):
            ds = scores[i]
            if not np.any(np.isfinite(ds)):
                cur = -1; held = 0; continue
            if cur == -1 or not np.isfinite(ds[cur]):
                cur = int(np.argmax(ds)); held = 1
            else:
                best = int(np.argmax(ds))
                if best != cur and held >= min_hold:
                    if stick == 'soft':
                        switch = ds[best] >= ds[cur] + 0.10
                    else:  # hard: switch only if cur < 0.90 * best
                        switch = ds[cur] < 0.90 * ds[best]
                    if switch:
                        cur = best; held = 1
                    else: held += 1
                else: held += 1
            win[i] = cur
        winners = win
    rows = []
    for i, d in enumerate(idx):
        c_ix = int(winners[i])
        if c_ix < 0: continue
        rzv = resz[i, c_ix]
        if not np.isfinite(rzv): continue
        rows.append({
            'date': d, 'sector': sector, 'target': cand_arr[c_ix],
            'residual_z': float(rzv),
            'predicted_return': float(pr[i, c_ix]) if pd.notna(pr[i, c_ix]) else 0.0,
            'next_ret': float(nxt[i, c_ix]) if pd.notna(nxt[i, c_ix]) else 0.0,
        })
    return pd.DataFrame(rows)

def build_panel_all(score_fn, universe, stick):
    parts = [build_panel(s, score_fn, universe, stick) for s in STACKS]
    return pd.concat(parts, ignore_index=True).sort_values(['date','sector']).reset_index(drop=True)

print('selector ready.')

In [ ]:
# Cell 8 — Simple PM (no regime classifier): mean-reversion on residual_z, force-exit on target switch
PM_COST_BPS = 5.0/1e4
RZ_ENTRY    = 1.5
RZ_EXIT     = 0.30
MAX_HOLD    = 10
STOP_LOSS   = -0.02
TAKE_PROFIT =  0.03

def pm_simulate(panel, rz_thr=RZ_ENTRY, rz_exit=RZ_EXIT, max_hold=MAX_HOLD,
                stop=STOP_LOSS, take=TAKE_PROFIT, cost_bps=PM_COST_BPS):
    state = {}; rows = []
    for (date, sector), grp in panel.groupby(['date','sector'], sort=True):
        r = grp.iloc[0]
        st = state.get(sector, dict(position=0, days_in=0, trade_pnl=0.0, target=None))
        prev_pos = st['position']
        rz = float(r['residual_z']); nr = float(r['next_ret'])
        desired = prev_pos; action = 'HOLD'
        if prev_pos != 0 and st['target'] != r['target']:
            desired = 0; action = 'EXIT_SWITCH'
        elif prev_pos != 0:
            st['days_in'] += 1
            if (st['days_in'] >= max_hold or abs(rz) <= rz_exit
                or st['trade_pnl'] <= stop or st['trade_pnl'] >= take):
                desired = 0; action = 'EXIT'
        if desired == 0 and abs(rz) >= rz_thr:
            desired, action = (1, 'ENTER_LONG') if rz < 0 else (-1, 'ENTER_SHORT')
        turnover = abs(desired - prev_pos)
        gross = desired * nr; net = gross - turnover * cost_bps
        is_entry = action.startswith('ENTER'); is_exit = action.startswith('EXIT')
        if is_entry:
            st = dict(position=desired, days_in=1, trade_pnl=net, target=r['target'])
        elif is_exit:
            st = dict(position=0, days_in=0, trade_pnl=0.0, target=None)
        elif desired != 0:
            st['trade_pnl'] += net
        state[sector] = st
        rows.append({'date': date, 'sector': sector, 'target': r['target'],
                     'residual_z': rz, 'action': action,
                     'position': float(desired), 'turnover': float(turnover),
                     'net_pnl': float(net),
                     'is_entry': is_entry, 'is_exit': is_exit})
    return pd.DataFrame(rows)

def portfolio_metrics(td, ppy=252):
    if td.empty: return pd.Series(dtype=float)
    daily = td.groupby('date')['net_pnl'].mean().sort_index()
    if daily.empty: return pd.Series(dtype=float)
    eq = (1+daily).cumprod(); cum = float(eq.iloc[-1]-1.0); n = len(daily)
    ann_ret = float((1+cum)**(ppy/max(n,1))-1.0)
    ann_vol = float(daily.std(ddof=1)*math.sqrt(ppy)) if n>1 else np.nan
    sh = ann_ret/ann_vol if ann_vol and np.isfinite(ann_vol) and ann_vol!=0 else np.nan
    dn = daily[daily<0]
    dnv = float(dn.std(ddof=1)*math.sqrt(ppy)) if len(dn)>1 else np.nan
    so = ann_ret/dnv if dnv and np.isfinite(dnv) and dnv!=0 else np.nan
    peak = eq.cummax(); mdd = float((eq/peak-1.0).min())
    entries = td[td['is_entry']]
    return pd.Series({
        'days': n, 'entries': int(len(entries)),
        'long':  int((entries['position']==1).sum()),
        'short': int((entries['position']==-1).sum()),
        'cum_ret': round(cum,4), 'ann_ret': round(ann_ret,4),
        'sharpe': round(sh,4) if np.isfinite(sh) else np.nan,
        'sortino': round(so,4) if np.isfinite(so) else np.nan,
        'max_dd': round(mdd,4),
        'win_rate_days': round(float((daily>0).mean()),4),
    })
print('PM ready.')

In [ ]:
# Cell 9 — Run all 18 variants on TRAIN + VAL
SCORES   = ['rz', 'rz_x_q', 'q']
UNIVERSE = ['all', 'top3']
STICKY   = ['none', 'soft', 'hard']
VARIANTS = list(product(SCORES, UNIVERSE, STICKY))
print(f'Running {len(VARIANTS)} variants × 2 splits...')

rows = []; PANELS = {}
t0 = time.time()
for score_fn, uni, stick in VARIANTS:
    tag = f'{score_fn}|{uni}|{stick}'
    panel = build_panel_all(score_fn, uni, stick)
    PANELS[tag] = panel
    for split_name, widx in [('TRAIN', train_idx), ('VAL', val_idx)]:
        pan = panel[panel['date'].isin(widx)].copy()
        td = pm_simulate(pan)
        m = portfolio_metrics(td)
        n_switches = sum(
            (panel[panel['sector']==s].sort_values('date')['target']
                  .ne(panel[panel['sector']==s].sort_values('date')['target'].shift()).sum())
            for s in SECTOR_POOLS
        )
        rows.append({'split': split_name, 'score': score_fn, 'universe': uni,
                     'stick': stick, 'switches_total': n_switches, **m.to_dict()})
    print(f'  {tag:30s} done')
summary = pd.DataFrame(rows)
summary.to_csv(CACHE_DIR / 'variant_summary.csv', index=False)
print(f'\nElapsed: {time.time()-t0:.1f}s | {len(summary)} rows')

In [ ]:
# Cell 10 — Show results: TRAIN and VAL sorted by Sharpe
for split_name in ['TRAIN', 'VAL']:
    sub = summary[summary['split']==split_name].copy()
    sub = sub.sort_values('sharpe', ascending=False)
    print(f'\n=== {split_name} — 18 variants sorted by Sharpe ===')
    cols = ['score','universe','stick','entries','long','short',
            'sharpe','sortino','cum_ret','max_dd','win_rate_days','switches_total']
    print(sub[cols].to_string(index=False))

In [ ]:
# Cell 11 — Heatmap: VAL Sharpe by (score x stick) for each universe
val = summary[summary['split']=='VAL']
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, uni in zip(axes, UNIVERSE):
    pv = val[val['universe']==uni].pivot(index='score', columns='stick', values='sharpe')
    pv = pv.reindex(index=SCORES, columns=STICKY)
    arr = pv.values.astype(float)
    vmax = float(np.nanmax(np.abs(arr))) if np.isfinite(arr).any() else 1.0
    im = ax.imshow(arr, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set_xticks(range(len(STICKY))); ax.set_xticklabels(STICKY)
    ax.set_yticks(range(len(SCORES))); ax.set_yticklabels(SCORES)
    ax.set_title(f'VAL Sharpe — universe={uni}')
    for i in range(len(SCORES)):
        for j in range(len(STICKY)):
            v = arr[i,j]
            if np.isfinite(v):
                ax.text(j, i, f'{v:+.2f}', ha='center', va='center', fontsize=10,
                        color='black' if abs(v)<vmax*0.5 else 'white')
    fig.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 12 — Axis-effect summary: marginal impact of each axis (mean Sharpe on VAL)
val = summary[summary['split']=='VAL']
print('=== Marginal effect on VAL Sharpe (mean across the other axes) ===')
print('\nScore:');    print(val.groupby('score')['sharpe'].agg(['mean','min','max']).round(3))
print('\nUniverse:'); print(val.groupby('universe')['sharpe'].agg(['mean','min','max']).round(3))
print('\nStick:');    print(val.groupby('stick')['sharpe'].agg(['mean','min','max']).round(3))

best_val = val.sort_values('sharpe', ascending=False).head(3)
print('\n=== Top-3 on VAL ===')
print(best_val[['score','universe','stick','entries','sharpe','sortino','cum_ret','max_dd']].to_string(index=False))

In [ ]:
# Cell 13 — TEST the single best VAL variant (one shot, no further tuning)
val = summary[summary['split']=='VAL'].sort_values('sharpe', ascending=False)
best = val.iloc[0]
tag = f"{best['score']}|{best['universe']}|{best['stick']}"
print(f'TEST variant: {tag}')
panel = PANELS[tag]
test_pan = panel[panel['date'].isin(test_idx)].copy()
td_test  = pm_simulate(test_pan)
m_test   = portfolio_metrics(td_test)
print('\n=== TEST metrics ===')
print(m_test.to_string())

train_row = summary[(summary['split']=='TRAIN') & (summary['score']==best['score'])
                    & (summary['universe']==best['universe']) & (summary['stick']==best['stick'])].iloc[0]
print('\n=== Same variant across splits ===')
comp = pd.DataFrame({
    'TRAIN': {k: train_row.get(k) for k in ['entries','long','short','sharpe','sortino','cum_ret','max_dd']},
    'VAL':   {k: best.get(k)      for k in ['entries','long','short','sharpe','sortino','cum_ret','max_dd']},
    'TEST':  {k: m_test.get(k)    for k in ['entries','long','short','sharpe','sortino','cum_ret','max_dd']},
})
print(comp.to_string())

# Log to run_history
RUN_HISTORY = Path('outputs/tuning_cache/run_history.csv')
RUN_HISTORY.parent.mkdir(parents=True, exist_ok=True)
HIST_COLS = ['timestamp','source','tag','clf_trial','f1_val','rz_thr','conf','mr_exit',
             'total_entries','sharpe','sortino','cumulative_return',
             'annualized_return','annualized_volatility','max_drawdown',
             'win_rate_days','active_days']
ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
row = {'timestamp': ts, 'source': 'DBTS_Variants_TEST', 'tag': tag,
       'clf_trial': None, 'f1_val': None,
       'rz_thr': RZ_ENTRY, 'conf': None, 'mr_exit': RZ_EXIT,
       'total_entries': int(m_test.get('entries',0)),
       'sharpe': m_test.get('sharpe'), 'sortino': m_test.get('sortino'),
       'cumulative_return': m_test.get('cum_ret'),
       'annualized_return': m_test.get('ann_ret'),
       'annualized_volatility': None,
       'max_drawdown': m_test.get('max_dd'),
       'win_rate_days': m_test.get('win_rate_days'),
       'active_days': None}
new = pd.DataFrame([{c: row.get(c) for c in HIST_COLS}])
if RUN_HISTORY.exists():
    prev = pd.read_csv(RUN_HISTORY); combined = pd.concat([prev, new], ignore_index=True)
else:
    combined = new
combined.to_csv(RUN_HISTORY, index=False)
print(f'\nLogged TEST row. Total history: {len(combined)}')

In [ ]:
# Cell 14 — Equity curves for top-3 VAL variants (TRAIN/VAL/TEST)
top3 = summary[summary['split']=='VAL'].sort_values('sharpe', ascending=False).head(3)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, split_name, widx in zip(axes, ('TRAIN','VAL','TEST'), (train_idx, val_idx, test_idx)):
    for _, r in top3.iterrows():
        tag = f"{r['score']}|{r['universe']}|{r['stick']}"
        pan = PANELS[tag]
        sub = pan[pan['date'].isin(widx)]
        td  = pm_simulate(sub)
        if td.empty: continue
        daily = td.groupby('date')['net_pnl'].mean().sort_index()
        eq = (1+daily).cumprod()
        ax.plot(eq.index, eq.values, label=tag, linewidth=1.1)
    ax.axhline(1.0, color='black', linewidth=0.4)
    ax.set_title(f'{split_name} — top-3 VAL variants')
    ax.grid(alpha=0.3); ax.legend(fontsize=7, loc='best')
plt.tight_layout(); plt.show()